# 05 — Trading agents

**Phase 5 deliverable:** all 23 agents backtested on KBANK, frictionless and with frictions.

23 notebooks, 6 families. The Q-learning set alone is 11 notebooks sharing one replay-buffer
skeleton with a swapped head — implemented here as `{double, duel, recurrent, curiosity}` flags
that compose.

## Two corrections to upstream, both load-bearing

**Agents get a real holdout.** In `agent/6.evolution-strategy-agent.ipynb`, `get_reward()` (the
training objective) and `buy()` (the reported equity curve) iterate the *same* `self.trend`. Every
published agent return in that repository is in-sample. Here `fit` sees the training block and the
reported result comes from the test block that follows it.

**The `close[t]` global is fixed, not ported.** Notebook 6's buy branch reads
`starting_money -= close[t]` — a module-level global — where the sell branch correctly reads
`self.trend[t]`. It is silent only because the two happen to hold the same list, and it detonates
the first time two tickers share a process. Nothing here reads a global: cash and inventory live
on the environment during training and on `SETMarket` during evaluation.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

In [ ]:
from stock_retrofit.config import all_agent_specs
from stock_retrofit.agents import registered_kinds

print("agent families:", ", ".join(registered_kinds()), "\n")
for spec in all_agent_specs():
    print(f"  {spec.name:36s} {spec.kind:20s} <- {spec.upstream}")

## How the 11 Q-learning notebooks become one class

| flag | what it does |
|---|---|
| `double` | the online net picks the next action, the target net scores it — removes the max operator's optimism bias |
| `duel` | the head splits into value and advantage streams recombined as `V + (A − mean A)` |
| `recurrent` | an LSTM reads the window as a sequence instead of an MLP reading it flattened |
| `curiosity` | a forward model's prediction error is added to the reward as an exploration bonus |

In [ ]:
specs = {s.name: s for s in all_agent_specs()}
rows = []
for name, spec in specs.items():
    if spec.kind == "q_learning":
        p = spec.params
        rows.append({"config": name,
                     "double": p.get("double"), "duel": p.get("duel"),
                     "recurrent": p.get("recurrent"), "curiosity": p.get("curiosity"),
                     "upstream": spec.upstream.split("/")[-1]})
pd.DataFrame(rows)

## A note on training speed

Learning agents need tens of thousands of simulated steps per fold, so **training** runs against a
fast numpy environment with a proportional round-trip cost. **Evaluation never does** — every
reported number comes from replaying the trained policy through `SETMarket` with board lots, tick
snapping, commission + VAT, price limits and the participation cap all enforced.

Training may use any objective it likes. Only the evaluation is a claim.

## Run the whole catalogue

Equivalent to:

```bash
python -m stock_retrofit.cli backtest --all --symbol KBANK
```

This takes roughly 20 minutes on CPU — the recurrent Q-learning variants dominate.

In [ ]:
from stock_retrofit.paths import RESULTS_DIR
from stock_retrofit.report import backtest_symbol

cached = RESULTS_DIR / "backtest-KBANK.csv"
agents = pd.read_csv(cached) if cached.exists() else backtest_symbol("KBANK")
agents

## Reading the result

`ret_frictionless` vs `ret_friction` is the headline (spec R11). The gap is what it costs to stop
pretending the market has no board lots, no tick sizes, no commission and no VAT.

Buy-and-hold is pinned to the top as the agent baseline — the counterpart to `NaiveLag` on the
model tables. An agent that trades hard and lands below it has bought turnover, not alpha.

In [ ]:
ok = agents[agents["status"] == "ok"]
print(f"profitable frictionless : {int((ok['ret_frictionless'] > 0).sum())} of {len(ok)}")
print(f"profitable after costs  : {int((ok['ret_friction'] > 0).sum())} of {len(ok)}")
print(f"mean cost of frictions  : {ok['friction_gap'].mean():+.2%} per fold")

baseline = ok.loc[ok["agent"].str.contains("buy_and_hold"), "ret_friction"].iloc[0]
beat = ok[(ok["ret_friction"] > baseline) & (~ok["agent"].str.contains("buy_and_hold"))]
print(f"\nbeat buy-and-hold after costs: {len(beat)} of {len(ok) - 1}")
ok.nlargest(8, "friction_gap")[["agent", "trades", "ret_frictionless", "ret_friction", "friction_gap"]]